In [1]:
import torch
from monai.metrics import DiceMetric
from monai.losses import DiceLoss


In [3]:
dice = DiceMetric(include_background=True,reduction="mean_batch",ignore_empty=False)

In [2]:
dice_loss = DiceLoss(sigmoid=False)
dice_loss_batch = DiceLoss(sigmoid=False, batch=True)

In [3]:
def complement_randomly_chosen_elements(y_true, frac=0.3):
    assert ((y_true >= 0) & (y_true <= 1)).all()
    #create y_pred by randomly swapping x% of y_true values
    y_pred = y_true.detach().clone()
    num_elements = y_true.numel()
    num_samples = int(num_elements * frac)  # x% of the elements
    #swap randomly chosen indices
    idxs = torch.randperm(num_elements)[:num_samples]
    idxs = torch.unravel_index(idxs,shape=y_true.shape)
    y_pred[idxs] = 1 - y_pred[idxs]
    return y_pred

In [4]:
y_true = torch.rand(1,3,128,128,128)
y_pred = complement_randomly_chosen_elements(y_true, frac=0.1)
y_true = y_true.gt(0.9)


In [5]:
print(dice_loss_batch(y_pred,y_true))
print(dice_loss(y_pred,y_true))


tensor(0.7130)
tensor(0.7130)


In [58]:
from src.utils.model_utils import window2patches
dice(window2patches(y_pred),window2patches(y_true))
dice.aggregate()

tensor([0.4773, 0.4774, 0.4774])

In [6]:
from src.utils.model_utils import window2patches


In [7]:
dice_loss(window2patches(y_pred),window2patches(y_true))

tensor(0.7131)

In [8]:
dice_loss_batch(window2patches(y_pred),window2patches(y_true))

tensor(0.7130)

Observation: Patch wise dice loss is same as image wise dice, but patch wise **dice metric is not**